In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, mean_squared_error

import torch.nn as nn
import torch.optim as optim
import math
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader
import datetime

import random
from skopt import BayesSearchCV
from sklearn.metrics import r2_score
from sklearn.model_selection import PredefinedSplit

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

In [4]:
data = pd.read_excel('../Data/New_data11.xlsx')
data_pop = pd.read_excel('../Data/population_2013_2072.xlsx')

In [5]:
for i in range(2013,2025):
    data_pop.loc[data_pop['year']==i,0] = data.loc[data['Year']==i,'Population_0'].iloc[0]
    data_pop.loc[data_pop['year']==i,40] = data.loc[data['Year']==i,'Population_40'].iloc[0]
    data_pop.loc[data_pop['year']==i,60] = data.loc[data['Year']==i,'Population_60'].iloc[0]

In [6]:
Climber = pd.read_excel('../Data/Climber.xlsx')

In [7]:
chuseok_dates = pd.read_excel('../Data/추석날짜.xlsx')

In [8]:
chuseok_dates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    57 non-null     datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 584.0 bytes


In [9]:
chuseok_week = []
# 각 해의 추석 날짜가 몇 번째 주에 속하는지 계산
for date_str in chuseok_dates['date'].values:
    chuseok_date = date_str.astype('datetime64[s]').astype(datetime.datetime)
    chuseok_week.append(chuseok_date.isocalendar()[1])  # ISO 캘린더에서 주차 계산
print(chuseok_week)

[37, 39, 37, 40, 39, 37, 40, 38, 36, 39, 38, 41, 39, 37, 40, 38, 37, 40, 38, 36, 39, 37, 40, 39, 37, 39, 38, 37, 39, 38, 40, 39, 37, 40, 39, 36, 39, 38, 36, 39, 38, 40, 38, 37, 40, 38, 37, 39, 37, 40, 39, 38, 39, 38, 37, 39, 38]


In [10]:
data['Chuseok'] = 0

In [11]:
for j, i in enumerate(range(2014,2025)):
    data.loc[(data['Year']==i) & (data['Week']>=chuseok_week[j]-2) & (data['Week']<=chuseok_week[j]+2), 'Chuseok']=1

In [12]:
data['Elder_cases'] = data['Cases_60']
data['Elder_incidence'] = 0.0

data['Nonelder_cases'] = data['Cases_0']+data['Cases_40']
data['Nonelder_incidence'] = 0.0

In [13]:
for i in range(2013,2025):
    data.loc[data['Year']==i,'Elder_incidence'] = 1000000*data.loc[data['Year']==i,'Elder_cases']/data_pop.loc[data_pop['year']==i,60].values[0]
    data.loc[data['Year']==i,'Nonelder_incidence'] = 1000000*data.loc[data['Year']==i,'Nonelder_cases']/(data_pop.loc[data_pop['year']==i,40].values[0] + data_pop.loc[data_pop['year']==i,0].values[0])

In [14]:
observed_year_incidence = pd.DataFrame(columns=['year','Total incidence','Elder incidence','Nonelder incidence'])
observed_year_cases = pd.DataFrame(columns=['year','Total cases','Elder cases','Nonelder cases'])
for num, i in enumerate(range(2015,2025)):
    observed_year_incidence.loc[num,'year'] = i
    observed_year_cases.loc[num,'year'] = i
    observed_year_cases.loc[num,'Total cases'] = data.loc[(data['Year']==i),'Cases'].sum()
    observed_year_incidence.loc[num,'Total incidence'] = 1000000*observed_year_cases.loc[num,'Total cases']/data_pop.loc[data_pop['year']==i,[0,40,60]].sum(axis=1).values[0]
    observed_year_incidence.loc[num,'Elder incidence'] = data.loc[(data['Year']==i),'Elder_incidence'].sum()
    observed_year_incidence.loc[num,'Nonelder incidence'] = data.loc[(data['Year']==i),'Nonelder_incidence'].sum()
    observed_year_cases.loc[num,'Elder cases'] = data.loc[(data['Year']==i),'Elder_cases'].sum()
    observed_year_cases.loc[num,'Nonelder cases'] = data.loc[(data['Year']==i),'Nonelder_cases'].sum()

In [15]:
observed_year_incidence['rate']=observed_year_incidence['Elder incidence']/observed_year_incidence['Nonelder incidence']

In [16]:
observed_year_incidence

,year,Total incidence,Elder incidence,Nonelder incidence,rate
0,2015,1.548566,5.811451,0.599191,9.698835
1,2016,3.221536,11.485501,1.278144,8.986078
2,2017,5.295753,18.365782,2.042529,8.991686
3,2018,5.020834,17.0151,1.861381,9.141117
4,2019,4.307945,14.668272,1.409221,10.408781
5,2020,4.68784,14.540912,1.730624,8.402119
6,2021,3.322417,10.368342,1.047658,9.89669
7,2022,3.735057,11.424875,1.115628,10.24076
8,2023,3.828853,11.82998,0.946905,12.493313
9,2024,3.30428,10.106488,0.744646,13.572198


In [17]:
data['Weekly hiker'] = data['Weekly hiker']*(data['Population_60']/data['Population'])

In [18]:
start_year = 2015
end_year = 2023

data_train = data[(data['Year']>=start_year) & (data['Year']<end_year)]
data_test = data[data['Year']>=end_year]

In [19]:
def make_dataset_D(x_data, y_data, window_size):
    x_list = []
    y_list = []
    for i in range(len(x_data) - window_size+1):
        x_list.append(np.array(x_data.iloc[i:i+window_size]))
        y_list.append(np.array(y_data.iloc[i+window_size-1]))
    x_list = np.array(x_list)
    y_list = np.array(y_list).reshape(-1)
    return x_list, y_list

In [20]:
features = ['tem','rain', 'hum', 'Chuseok', 'Weekly hiker', 'Tick Density']

In [21]:
target = ['Elder_incidence']

In [22]:
import random
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [23]:
test_num = 105
valid_num = 52

seed_value = 42

In [24]:
for window_size in range(2,11):
    data_temp = data_train[(data_train['Year']>start_year) | (data_train['Week']>53-window_size+1)]
    data_temp.index = range(len(data_temp))
    data_test.index = range(len(data_test))
    X = pd.concat([data_temp.loc[:, features],data_test.loc[:, features]])
    Y = pd.concat([data_temp[target],data_test[target]])
    X.index=range(len(X))
    Y.index=range(len(Y))
    X_max = X[:-test_num].max()
    X_min = X[:-test_num].min()
    X_s = (X-X_min)/(X_max-X_min)
    
    train_num = len(X) - valid_num - test_num - window_size + 1
    temp_X = X_s.copy()
    temp_Y = Y.copy()
        
    temp_X_w, temp_Y_w = make_dataset_D(temp_X, temp_Y, window_size)
    temp_X_w_1 = temp_X_w.reshape(temp_X_w.shape[0],-1)
    
    X_train = temp_X_w_1[:train_num]
    X_valid = temp_X_w_1[train_num:train_num+valid_num]
    X_test = temp_X_w_1[-test_num:]
    
    y_train = temp_Y_w[:train_num]
    y_valid = temp_Y_w[train_num:train_num+valid_num]
    y_test = temp_Y_w[-test_num:]
    
    X_train_total = np.vstack([X_train, X_valid])
    y_train_total = np.concatenate([y_train, y_valid])
    
    test_fold = np.array([-1]*len(X_train) + [0]*len(X_valid))
    ps = PredefinedSplit(test_fold)
    
    rmse_scorer = make_scorer(
            lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),  # ← squared 대신 직접 루트
            greater_is_better=False
        )
    np.random.seed(seed_value)
    random.seed(seed_value)
    
    LR_model = LinearRegression()
    LR_model.fit(X_train_total, y_train_total)
    
    LR_pred_train = LR_model.predict(X_train_total)
    LR_pred_test = LR_model.predict(X_test)
    
    LR_pred_train = np.maximum(0, LR_pred_train)
    LR_pred_test = np.maximum(0, LR_pred_test)
    
    np.random.seed(seed_value)
    random.seed(seed_value)
    
    random_forest_model = RandomForestRegressor(random_state=seed_value)
    # param_grid_rf = {
    #     'n_estimators': (100, 500),  # 트리 개수 범위
    #     'max_features': ['sqrt', 'log2'],  # 특성 선택 기준
    #     'max_depth': (30, 70),  # 트리 최대 깊이 범위
    #     'min_samples_split': (5, 15),  # 노드 분할 기준 범위
    #     'min_samples_leaf': (2, 6),  # 리프 노드 최소 샘플 수 범위
    #     'bootstrap': [True]  # 부트스트랩 샘플링 유지
    # }
    param_grid_rf = {
        'n_estimators': (100, 1000),  # 트리 개수 범위
        'max_features': ['sqrt', 'log2'],  # 특성 선택 기준
        'max_depth': (2, 70),  # 트리 최대 깊이 범위
        'min_samples_split': (3, 15),  # 노드 분할 기준 범위
        'min_samples_leaf': (2, 10),  # 리프 노드 최소 샘플 수 범위
        'bootstrap': [True]  # 부트스트랩 샘플링 유지
    }
    
    Bays_rf = BayesSearchCV(
        estimator=random_forest_model,
        search_spaces=param_grid_rf,
        scoring='neg_mean_squared_error',  # MSE 사용
        cv=5,                # 5-fold 교차 검증
        n_iter=100,          # 최대 100번의 탐색
        random_state=seed_value,
        verbose=1,           # 탐색 진행 상황 출력
        n_jobs=-1            # 병렬 처리
    )
    
    Bays_rf.fit(X_train_total, y_train_total)
    
    RF_model = RandomForestRegressor(random_state=seed_value, **Bays_rf.best_params_)
    RF_model.fit(X_train_total, y_train_total)
    
    np.random.seed(seed_value)
    random.seed(seed_value)
    # XGBoost 모델 생성
    XGB_model = XGBRegressor(
        base_score=float(np.mean(y_train_total)),  # 타깃 평균으로 초기값 보정
        random_state=seed_value
    )
    
    # 베이지안 최적화를 위한 하이퍼파라미터 범위 설정 (과적합 방지 적용)
    XGB_params = {
        'learning_rate': (0.01, 0.2, 'log-uniform'),  # 학습률 (최적화 안정성 고려)
        'n_estimators': (100, 1000),                    # 추정기 수 (과적합 방지)
        'max_depth': (3, 10),                          # 최대 깊이 (깊이 제한)
        'colsample_bytree': (0.3, 0.9),               # 열 샘플링 비율 (과적합 방지)
        'subsample': (0.4, 0.9)                       # 데이터 샘플링 비율 (일반화 향상)
    }
    
    
    # 베이지안 최적화 객체 생성
    Bays_xgb = BayesSearchCV(
        estimator=XGB_model,
        search_spaces=XGB_params,
        scoring=rmse_scorer,  # 사용자 정의 MSE 스코어러
        cv=ps,                # 5-fold 교차 검증
        n_iter=100,          # 최대 100번의 탐색
        random_state=seed_value,
        verbose=0            # 탐색 진행 상황 출력
    )
    Bays_xgb.fit(X_train_total, y_train_total)
    
    XGB_model = XGBRegressor(base_score=float(np.mean(y_train_total)),
                             learning_rate=Bays_xgb.best_params_['learning_rate'],
                          n_estimators=Bays_xgb.best_params_['n_estimators'], 
                          max_depth=Bays_xgb.best_params_['max_depth'],
                          colsample_bytree=Bays_xgb.best_params_['colsample_bytree'],
                          subsample=Bays_xgb.best_params_['subsample'],
                          random_state=42)
    XGB_model.fit(X_train_total, y_train_total)

    joblib.dump(LR_model, './hyperparameter_2/LR_model_Regression_'+str(window_size)+'.pkl')
    joblib.dump(RF_model, './hyperparameter_2/RF_model_Regression_'+str(window_size)+'.pkl')
    XGB_model.save_model(f'./hyperparameter_2/XGB_model_Regression_{window_size}.json')

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi